# Conn2Conn Kraken Prediction Evaluation

This notebook re-runs the default evaluation workflow against the current codebase. It keeps the old high-level structure: shared setup, closed-form baselines, learned models, and Krakencoder precomputed predictions. Results are written under `results/local_results/<model_name>/final/` with a local artifact layout mirroring the Tune outputs, plus an `eval_test.md` report for each model.

Notes:
- Default model configs from `models/configs/*.yml` are used.
- The notebook caches dataset bundles by data spec so HCP loading is not repeated for every model.
- Learned models will use their full default training settings, so those sections may take a while.

In [1]:
from pathlib import Path
from copy import deepcopy
import json
import sys

import numpy as np
import pandas as pd
import torch
from scipy.io import loadmat
from torch.utils.data import DataLoader

REPO_ROOT = Path('/scratch/asr655/neuroinformatics/Conn2Conn')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.hcp_dataset import HCP_Base, HCP_Partition
from results.local_results_utils import best_local_results, load_local_results, plot_metric_bar, plot_metric_scatter
from models.config import build_model, get_default_config, load_config, resolve_source_dependent_config
from models.eval import Evaluator
from models.loss import train_model
from models.models import predict_from_loader

LOCAL_RESULTS_ROOT = REPO_ROOT / 'results' / 'local_results'
LOCAL_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

CLOSED_FORM_MODELS = [
    'CrossModalPCA',
    'CrossModal_PLS_SVD',
    'CrossModal_PCA_PLS',
]

LEARNED_MODELS = [
    'CrossModal_PCA_PLS_learnable',
    'CrossModal_PCA_PLS_FSResidual',
    'CrossModalVAE',
]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Local results root:', LOCAL_RESULTS_ROOT)

Device: cuda
Local results root: /scratch/asr655/neuroinformatics/Conn2Conn/results/local_results


In [2]:
DATA_CACHE = {}


def to_jsonable(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def default_data_config(cfg):
    base = {
        'parcellation': 'Glasser',
        'hemi': 'both',
        'source': 'SC',
        'target': 'FC',
        'shuffle_seed': 0,
    }
    base.update(deepcopy(cfg.get('data', {})))
    return base


def get_dataset_bundle(data_cfg, batch_size=128):
    resolved = {
        'parcellation': data_cfg.get('parcellation', 'Glasser'),
        'hemi': data_cfg.get('hemi', 'both'),
        'source': data_cfg.get('source', 'SC'),
        'target': data_cfg.get('target', 'FC'),
        'shuffle_seed': data_cfg.get('shuffle_seed', 0),
    }
    key = (
        resolved['parcellation'],
        resolved['hemi'],
        resolved['source'],
        resolved['target'],
        resolved['shuffle_seed'],
        batch_size,
    )
    if key not in DATA_CACHE:
        base = HCP_Base(**resolved)
        train_ds = HCP_Partition(base, 'train')
        val_ds = HCP_Partition(base, 'val')
        test_ds = HCP_Partition(base, 'test')
        bundle = {
            'base': base,
            'train_ds': train_ds,
            'val_ds': val_ds,
            'test_ds': test_ds,
            'train_loader': DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            'val_loader': DataLoader(val_ds, batch_size=batch_size, shuffle=False),
            'test_loader': DataLoader(test_ds, batch_size=batch_size, shuffle=False),
        }
        DATA_CACHE[key] = bundle
        print('Loaded dataset bundle:', key)
    return DATA_CACHE[key]


def get_modality_array(base, modality):
    if modality == 'SC':
        return base.sc_upper_triangles
    if modality == 'FC':
        return base.fc_upper_triangles
    if modality == 'SC_r2t':
        return base.sc_r2t_corr_upper_triangles
    raise ValueError(f'Unknown modality: {modality}')


def local_artifact_dir(model_name):
    out = LOCAL_RESULTS_ROOT / model_name / 'final'
    out.mkdir(parents=True, exist_ok=True)
    return out


def evaluate_and_store(model_name, cfg, learned, bundle, model, train_result=None, extra_config=None):
    artifact_dir = local_artifact_dir(model_name)
    report_base = artifact_dir / 'eval_test'

    train_preds, train_targets = predict_from_loader(model, bundle['train_loader'])
    val_preds, val_targets = predict_from_loader(model, bundle['val_loader'])
    test_preds, test_targets = predict_from_loader(model, bundle['test_loader'])

    train_eval = Evaluator(train_preds, train_targets, bundle['train_ds'], bundle['base'])
    val_eval = Evaluator(val_preds, val_targets, bundle['val_ds'], bundle['base'])
    test_eval = Evaluator(test_preds, test_targets, bundle['test_ds'], bundle['base'])
    test_metrics = test_eval.analyze_results(verbose=True, filepath=str(report_base), model_name=model_name)

    metrics_out = {
        'train': to_jsonable(train_eval._metrics),
        'val': to_jsonable(val_eval._metrics),
        'test': to_jsonable(test_metrics),
    }

    config_out = deepcopy(cfg)
    if extra_config:
        config_out.setdefault('extra', {}).update(extra_config)

    config_path = artifact_dir / 'config.json'
    metrics_path = artifact_dir / 'metrics_final.json'
    report_path = artifact_dir / 'eval_test.md'
    history_path = None

    if train_result is not None and hasattr(train_result, 'history_df'):
        history_path = artifact_dir / 'train_history.csv'
        train_result.history_df.to_csv(history_path, index=False)

    config_path.write_text(json.dumps(to_jsonable(config_out), indent=2))
    metrics_path.write_text(json.dumps(metrics_out, indent=2))

    manifest = {
        'trial_id': model_name,
        'learned': learned,
        'config_path': str(config_path),
        'model_path': None,
        'metrics_path': str(metrics_path),
        'report_path': str(report_path),
        'history_path': str(history_path) if history_path is not None else None,
    }
    (artifact_dir / 'artifact_manifest.json').write_text(json.dumps(manifest, indent=2))

    base_metrics = test_metrics.get('base_metrics', {}) if isinstance(test_metrics, dict) else {}
    summary_row = {
        'model': model_name,
        'learned': learned,
        'source': bundle['base'].source,
        'target': bundle['base'].target,
        'val_mse': val_eval._metrics.get('mse'),
        'val_demeaned_r': val_eval._metrics.get('demeaned_pearson'),
        'test_mse': base_metrics.get('mse'),
        'test_demeaned_r': base_metrics.get('demeaned_pearson'),
        'artifact_dir': str(artifact_dir),
    }
    return {
        'summary': summary_row,
        'config': config_out,
        'metrics': metrics_out,
        'artifact_dir': artifact_dir,
    }


def run_model_with_overrides(
    model_name,
    data_overrides=None,
    model_overrides=None,
    trainer_overrides=None,
    result_name=None,
):
    cfg = resolve_source_dependent_config(get_default_config(model_name))
    loaded_cfg = load_config(model_name)
    learned = bool(loaded_cfg.get('learned', True))

    cfg.setdefault('data', {})
    cfg.setdefault('model', {})
    cfg.setdefault('trainer', {})

    if data_overrides:
        cfg['data'].update(deepcopy(data_overrides))
    if model_overrides:
        cfg['model'].update(deepcopy(model_overrides))
    if trainer_overrides:
        cfg['trainer'].update(deepcopy(trainer_overrides))

    cfg = resolve_source_dependent_config(cfg)
    data_cfg = default_data_config(cfg)
    trainer_cfg = deepcopy(cfg.get('trainer', {}))
    batch_size = trainer_cfg.get('batch_size', 128)
    bundle = get_dataset_bundle(data_cfg, batch_size=batch_size)

    model_cfg = deepcopy(cfg.get('model', {}))
    model_cfg.pop('name', None)
    model = build_model(bundle['base'], model_name, model_cfg)

    train_result = None
    if learned:
        train_result = train_model(
            model,
            bundle['train_loader'],
            bundle['val_loader'],
            base=bundle['base'],
            log_every=trainer_cfg.get('log_every', 5),
            lr=trainer_cfg.get('lr', 1e-4),
            loss_type=trainer_cfg.get('loss_type', 'mse'),
            loss_alpha=trainer_cfg.get('loss_alpha', 0.5),
            loss_beta=trainer_cfg.get('loss_beta', 1.0),
            max_epochs=trainer_cfg.get('max_epochs', 100),
            logger=False,
            pl_logger=None,
        )
        model = train_result.pl_module.model

    eval_name = result_name or model_name
    out = evaluate_and_store(
        eval_name,
        cfg,
        learned,
        bundle,
        model,
        train_result=train_result,
        extra_config={'base_model_name': model_name},
    )
    out['summary']['base_model_name'] = model_name
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out


def run_default_model(model_name):
    return run_model_with_overrides(model_name)


def run_krakencoder_precomputed(parcellation='Glasser', source_conn_type='SC'):
    data_cfg = {
        'parcellation': parcellation,
        'hemi': 'both',
        'source': source_conn_type,
        'target': 'FC',
        'shuffle_seed': 0,
    }
    bundle = get_dataset_bundle(data_cfg, batch_size=128)
    base = bundle['base']

    mat_path = REPO_ROOT / 'krakencoder' / 'example_data' / f'mydata_kraken_source_{parcellation}.{source_conn_type}.mat'
    mat = loadmat(mat_path)
    preds = mat['predicted_alltypes'][0][0][0][0][0]
    kraken_fc_preds = np.array(preds[1])

    target_array = get_modality_array(base, 'FC')
    split_indices = base.trainvaltest_partition_indices

    train_preds = kraken_fc_preds[split_indices['train']]
    val_preds = kraken_fc_preds[split_indices['val']]
    test_preds = kraken_fc_preds[split_indices['test']]

    train_targets = target_array[split_indices['train']]
    val_targets = target_array[split_indices['val']]
    test_targets = target_array[split_indices['test']]

    model_name = 'Krakencoder_precomputed'
    artifact_dir = local_artifact_dir(model_name)
    report_base = artifact_dir / 'eval_test'

    train_eval = Evaluator(train_preds, train_targets, bundle['train_ds'], base)
    val_eval = Evaluator(val_preds, val_targets, bundle['val_ds'], base)
    test_eval = Evaluator(test_preds, test_targets, bundle['test_ds'], base)
    test_metrics = test_eval.analyze_results(verbose=True, filepath=str(report_base), model_name=model_name)

    config_out = {
        'data': data_cfg,
        'model': {
            'name': model_name,
            'prediction_source': str(mat_path),
            'prediction_key': 'predicted_alltypes[1] -> FCcorr_Glasser_hpf',
        },
    }
    metrics_out = {
        'train': to_jsonable(train_eval._metrics),
        'val': to_jsonable(val_eval._metrics),
        'test': to_jsonable(test_metrics),
    }

    config_path = artifact_dir / 'config.json'
    metrics_path = artifact_dir / 'metrics_final.json'
    report_path = artifact_dir / 'eval_test.md'
    config_path.write_text(json.dumps(config_out, indent=2))
    metrics_path.write_text(json.dumps(metrics_out, indent=2))
    manifest = {
        'trial_id': model_name,
        'learned': False,
        'config_path': str(config_path),
        'model_path': None,
        'metrics_path': str(metrics_path),
        'report_path': str(report_path),
        'history_path': None,
    }
    (artifact_dir / 'artifact_manifest.json').write_text(json.dumps(manifest, indent=2))

    base_metrics = test_metrics.get('base_metrics', {}) if isinstance(test_metrics, dict) else {}
    return {
        'summary': {
            'model': model_name,
            'learned': False,
            'source': source_conn_type,
            'target': 'FC',
            'val_mse': val_eval._metrics.get('mse'),
            'val_demeaned_r': val_eval._metrics.get('demeaned_pearson'),
            'test_mse': base_metrics.get('mse'),
            'test_demeaned_r': base_metrics.get('demeaned_pearson'),
            'artifact_dir': str(artifact_dir),
        },
        'config': config_out,
        'metrics': metrics_out,
        'artifact_dir': artifact_dir,
    }


def _safe_scalar(x):
    if isinstance(x, (np.integer, np.int32, np.int64)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (int, float, bool, str)):
        return x
    return None


def _flatten_metric_dict(d, prefix=''):
    flat = {}
    if not isinstance(d, dict):
        return flat

    for key, value in d.items():
        name = f'{prefix}{key}' if prefix else key

        if key == 'ranklist':
            rank_arr = np.asarray(value, dtype=float)
            if rank_arr.size:
                flat[f'{prefix}rank_mean'] = float(rank_arr.mean())
                flat[f'{prefix}rank_median'] = float(np.median(rank_arr))
                flat[f'{prefix}rank_std'] = float(rank_arr.std())
                flat[f'{prefix}rank_min'] = float(rank_arr.min())
                flat[f'{prefix}rank_max'] = float(rank_arr.max())
                flat[f'{prefix}rank_p25'] = float(np.percentile(rank_arr, 25))
                flat[f'{prefix}rank_p75'] = float(np.percentile(rank_arr, 75))
            continue

        if isinstance(value, dict):
            flat.update(_flatten_metric_dict(value, prefix=f'{name}_'))
            continue

        scalar = _safe_scalar(value)
        if scalar is not None:
            flat[name] = scalar

    return flat


def summarize_runs(run_outputs):
    rows = []
    for out in run_outputs:
        row = dict(out.get('summary', {}))
        metrics = out.get('metrics', {})
        row.update(_flatten_metric_dict(metrics.get('train', {}), prefix='train_'))
        row.update(_flatten_metric_dict(metrics.get('val', {}), prefix='val_'))
        row.update(_flatten_metric_dict(metrics.get('test', {}), prefix='test_'))
        rows.append(row)

    df = pd.DataFrame(rows)
    if not df.empty:
        preferred = [
            'model',
            'base_model_name',
            'learned',
            'source',
            'target',
            'val_mse',
            'val_demeaned_r',
            'test_base_metrics_mse',
            'test_base_metrics_r2',
            'test_base_metrics_pearson',
            'test_base_metrics_demeaned_pearson',
            'test_heatmaps_raw_top1_acc',
            'test_heatmaps_raw_avg_rank_percentile',
            'test_heatmaps_raw_rank_mean',
            'test_heatmaps_raw_rank_median',
            'test_heatmaps_demeaned_top1_acc',
            'test_heatmaps_demeaned_avg_rank_percentile',
            'test_heatmaps_demeaned_rank_mean',
            'test_heatmaps_demeaned_rank_median',
            'artifact_dir',
        ]
        cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
        df = df[cols].sort_values('model').reset_index(drop=True)
    return df


## Krakencoder precomputed predictions

This block uses the bundled precomputed Krakencoder SC-to-FC predictions and generates the same markdown report / local artifact bundle.

In [ ]:
krakencoder_output = run_krakencoder_precomputed(parcellation='Glasser', source_conn_type='SC')
krakencoder_summary = summarize_runs([krakencoder_output])
krakencoder_summary

## Summary

This combines all runs and lists the saved artifact folders.

In [ ]:
all_outputs = krakencoder_output
summary_df = summarize_runs(all_outputs)
summary_df

## Saved reports

In [ ]:
saved_reports = sorted(LOCAL_RESULTS_ROOT.glob('*/final/eval_test.md'))
pd.DataFrame({'report_md': [str(p) for p in saved_reports]})

## Inspect current local results

These cells read directly from `results/local_results`, so you can skip the model-run sections above when you only want to inspect the current saved state.

In [ ]:
local_results_df = load_local_results(LOCAL_RESULTS_ROOT)
local_results_df

In [ ]:
local_results_df.columns.tolist()

In [ ]:
# Define 'to_show' as a list of run_name strings to display
to_show = [
    "Krakencoder_precomputed",
]

# Subset local_results_df to only include rows with run_name in to_show
to_show_df = local_results_df[local_results_df["run_name"].isin(to_show)]

plot_metric_scatter(
    to_show_df,
    x='test_base_metrics_mse',
    y='test_base_metrics_demeaned_pearson',
    label_col='run_name',
    title='Best local runs: test demeaned Pearson vs MSE')

## Seed Split Generation

Generate family-preserving train/val/test splits for seeds 0–4 using `load_metadata` from the Conn2Conn data utilities.

For each seed this produces:
- `krakencoder/example_data/HCP-YA_dataset/participants_seed{seed}.tsv` — same columns as `participants.tsv` with updated `train_val_test`
- `krakencoder/example_data/subject_splits_957subj_seed{seed}.mat` — same format as `subject_splits_957subj.mat` (0-based int64 indices)

In [4]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import savemat

REPO_ROOT = Path('/scratch/asr655/neuroinformatics/Conn2Conn')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.dataset_utils import load_metadata

EXAMPLE_DATA_DIR = REPO_ROOT / 'krakencoder' / 'example_data'
DATASET_DIR      = EXAMPLE_DATA_DIR / 'HCP-YA_dataset'
PARTICIPANTS_TSV = DATASET_DIR / 'participants.tsv'

SEEDS = list(range(10)) 

# Load original participants.tsv once — provides participant_id, subject, age, sex columns
original_participants = pd.read_csv(PARTICIPANTS_TSV, sep='\t')

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Seed {seed}")
    print('='*60)

    # load_metadata handles seed=0 (use original split) and seed>0 (generate_train_val_test)
    metadata_df, _ = load_metadata(shuffle_seed=seed)

    # Build a subject -> split label lookup from the Conn2Conn metadata
    split_map = dict(zip(metadata_df['subject'].astype(int), metadata_df['train_val_test']))

    # ── participants_seed{seed}.tsv ────────────────────────────────────────────
    new_participants = original_participants.copy()
    new_participants['train_val_test'] = new_participants['subject'].astype(int).map(split_map)

    n_train = new_participants['train_val_test'].eq('train').sum()
    n_val   = new_participants['train_val_test'].eq('val').sum()
    n_test  = new_participants['train_val_test'].eq('test').sum()
    print(f"  Split → train: {n_train}, val: {n_val}, test: {n_test}")

    participants_out = DATASET_DIR / f'participants_seed{seed}.tsv'
    new_participants.to_csv(participants_out, sep='\t', index=False)
    print(f"  Saved: {participants_out.relative_to(REPO_ROOT)}")

    # ── subject_splits_957subj_seed{seed}.mat ─────────────────────────────────
    # subjects array: float64 IDs in the same order as participants.tsv
    subjects = new_participants['subject'].astype(int).tolist()
    subjects_array = np.array(subjects, dtype=np.float64)

    # 0-based index arrays (matching krakencoder convention; see create_subject_splits_957.py)
    subjidx_train = np.array(
        [i for i, s in enumerate(subjects) if split_map.get(s) == 'train'], dtype=np.int64)
    subjidx_val   = np.array(
        [i for i, s in enumerate(subjects) if split_map.get(s) == 'val'],   dtype=np.int64)
    subjidx_test  = np.array(
        [i for i, s in enumerate(subjects) if split_map.get(s) == 'test'],  dtype=np.int64)

    mat_dict = {
        'subjects':         subjects_array,
        'subjidx_train':    subjidx_train,
        'subjidx_val':      subjidx_val,
        'subjidx_test':     subjidx_test,
        'subjidx_retest':   np.array([], dtype=np.int64),
        'subjidx_bad_data': np.array([], dtype=np.int64),
    }

    mat_out = EXAMPLE_DATA_DIR / f'subject_splits_957subj_seed{seed}.mat'
    savemat(mat_out, mat_dict, format='5', do_compression=True)
    print(f"  Saved: {mat_out.relative_to(REPO_ROOT)}")
    print(f"  MAT   → train: {len(subjidx_train)}, val: {len(subjidx_val)}, test: {len(subjidx_test)}")

print("\nDone. All seed splits saved.")



Seed 0
  Split → train: 683, val: 79, test: 195
  Saved: krakencoder/example_data/HCP-YA_dataset/participants_seed0.tsv
  Saved: krakencoder/example_data/subject_splits_957subj_seed0.mat
  MAT   → train: 683, val: 79, test: 195

Seed 1
  Split → train: 683, val: 79, test: 195
  Saved: krakencoder/example_data/HCP-YA_dataset/participants_seed1.tsv
  Saved: krakencoder/example_data/subject_splits_957subj_seed1.mat
  MAT   → train: 683, val: 79, test: 195

Seed 2
  Split → train: 683, val: 79, test: 195
  Saved: krakencoder/example_data/HCP-YA_dataset/participants_seed2.tsv
  Saved: krakencoder/example_data/subject_splits_957subj_seed2.mat
  MAT   → train: 683, val: 79, test: 195

Seed 3
  Split → train: 683, val: 79, test: 195
  Saved: krakencoder/example_data/HCP-YA_dataset/participants_seed3.tsv
  Saved: krakencoder/example_data/subject_splits_957subj_seed3.mat
  MAT   → train: 683, val: 79, test: 195

Seed 4
  Split → train: 683, val: 79, test: 195
  Saved: krakencoder/example_data/H